# About Dataset


Fraud detection datasets are used to train machine learning models to identify and classify fraudulent activities or transactions. These datasets typically contain labeled examples of both fraudulent and non-fraudulent instances, allowing the model to learn patterns and characteristics associated with fraud.

Here is a description of a typical fraud detection dataset:

Features: The dataset includes various features or attributes that describe each instance. These features can be both numerical and categorical. Examples of common features include transaction amount, transaction type, location, time of day, user demographics, device information, and previous transaction history.

Labels: Each instance in the dataset is labeled as either fraudulent or non-fraudulent. The labels indicate whether the corresponding transaction or activity is fraudulent or legitimate.

Imbalance: Fraud detection datasets often suffer from class imbalance, where the number of fraudulent instances is significantly smaller than the number of non-fraudulent instances. This is because fraudulent activities are usually rare compared to legitimate activities. Addressing class imbalance is crucial to avoid biased model performance.

Anonymized Data: To protect privacy and confidentiality, the dataset may include anonymized or obfuscated features. Personally identifiable information (PII) such as names, addresses, and account numbers are typically removed or masked to prevent identification of individuals.

Training and Testing Split: The dataset is usually divided into two subsets: a training set and a testing set. The training set is used to train the machine learning model, while the testing set is used to evaluate the model's performance on unseen data. This split helps assess the generalization ability of the model.

Ground Truth: The dataset should have accurate and reliable ground truth labels for each instance, indicating whether it is fraudulent or not. These labels are crucial for training and evaluating the performance of the fraud detection model.

It's important to note that fraud detection datasets can vary in size, complexity, and specific domain focus. Different datasets may be available for different industries or types of fraud, such as credit card fraud, insurance fraud, or online transaction fraud. Additionally, datasets may come from different sources, such as financial institutions, e-commerce platforms, or cybersecurity companies.

When working with fraud detection datasets, it's essential to preprocess the data, handle missing values, perform feature engineering, and apply appropriate techniques to handle class imbalance, such as undersampling, oversampling, or generating synthetic samples.

https://www.kaggle.com/datasets/ashishkumarjayswal/froud-detection-dataset

# Understanding the problem

1. Data cleaning including missing values, outliers and multi-collinearity.
2. Describe your fraud detection model in elaboration.
3. How did you select variables to be included in the model?
4. Demonstrate the performance of the model by using best set of tools.
5. What are the key factors that predict fraudulent customer?
6. Do these factors make sense? If yes, How? If not, How not?
7. What kind of prevention should be adopted while company update its   infrastructure?
8. Assuming these actions have been implemented, how would you determine if they work?

# Import necessary libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from IPython import display
#from ydata_profiling import ProfileReport
from sklearn.preprocessing import LabelEncoder
from statsmodels.stats.outliers_influence import variance_inflation_factor
from imblearn.over_sampling import SMOTE
from sklearn.utils import shuffle
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.metrics import roc_curve, roc_auc_score, precision_recall_curve, auc

# LOAD DATASET

In [ ]:
data = pd.read_csv("F:/fraud detection project/Fraud1.csv")
# Randomly sample 10% of the data (you can adjust the fraction)
#data = data.sample(frac=0.01, random_state=42)

# Display the sampled data
data

In [ ]:
data.columns

In [ ]:
#profile report
#ProfileReport(data)

# 1) DATA EXPLORATORY 

1. Display the first 5 rows of the dataset.

In [ ]:
data.head()

2) Display the last 5 rows of the dataset.

In [ ]:
data.tail()

3. Get the shape of the dataset (rows, columns).

In [ ]:
data.shape
data.info()

4. Get the column names and their data types.

In [ ]:
data.dtypes

5. Check for missing values in each column.

In [ ]:
data.isnull().sum()

6. Get summary statistics for numerical columns.

In [ ]:
data.describe()

7. Count the number of unique values in each column.

In [ ]:
data.nunique()

8. check the distribution of the type column (transaction types).

In [ ]:
data['type'].value_counts()

9. Check the distribution of the isFraud column (fraud vs. non-fraud).

In [ ]:
data['isFraud'].value_counts()

10. Check the distribution of the isFlaggedFraud column.

In [ ]:
data['isFlaggedFraud'].value_counts()

## 2) Data Cleaning

11. Drop rows with missing values.

In [ ]:
data.dropna(inplace=True)

12. Fill missing values in numerical columns with the mean.

In [ ]:
# data.fillna(data.mean(), inplace=True)

13. Convert the type column to a categorical data type.

In [ ]:
data['type'] = data['type'].astype('category')

14. Remove duplicate rows from the dataset.

In [ ]:
data.drop_duplicates(inplace=True)


15. Rename the nameOrig column to sender.

In [ ]:
data.rename(columns={'nameOrig': 'sender'}, inplace=True)

16. Rename the nameDest column to receiver.

In [ ]:
data.rename(columns={'nameDest': 'receiver'}, inplace=True)

17. Convert the step column to represent days instead of hours (assuming each step is an hour)

In [ ]:
data['step'] = data['step'] / 24

18. Create a new column balanceChangeOrig representing the change in the originator's balance.

In [ ]:
data['balanceChangeOrig'] = data['newbalanceOrig'] - data['oldbalanceOrg']

19. Create a new column balanceChangeDest representing the change in the destination's balance.

In [ ]:
data['balanceChangeDest'] = data['newbalanceDest'] - data['oldbalanceDest']

20. Drop columns that are not needed for analysis (e.g., isFlaggedFraud).

In [ ]:
data.drop(columns=['isFlaggedFraud'], inplace=True)

In [ ]:
data

## 3) DATA ANALYSIS

21. Find the average transaction amount for fraudulent vs. non-fraudulent transactions.

In [ ]:
data.groupby('isFraud')['amount'].mean()

22. Find the total transaction amount for each transaction type.

In [ ]:
data.groupby('type')['amount'].sum()

23. Find the maximum transaction amount for fraudulent transactions.

In [ ]:
data[data['isFraud'] == 0]['amount'].min()

24. Find the minimum transaction amount for non-fraudulent transactions.

In [ ]:
data[data['isFraud'] == 1]['amount'].min()

25. Count the number of fraudulent transactions for each transaction type.

In [ ]:
data[data['isFraud'] == 1]['type'].value_counts()

26. Find the percentage of fraudulent transactions.



In [ ]:
(data['isFraud'].sum() / len(data)) * 100

27. Find the correlation between numerical columns.

Select Only Numerical Columns

In [ ]:
# Select only numerical columns
numerical_data = data.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
correlation_matrix = numerical_data.corr()

# Display the correlation matrix
print(correlation_matrix)

Encode Categorical Columns

Label Encoding

In [ ]:
print(data.columns)

In [ ]:
from sklearn.preprocessing import LabelEncoder

# Encode the 'type' column
label_encoder = LabelEncoder()
data['type_encoded'] = label_encoder.fit_transform(data['type'])

# Select all numerical columns (including the encoded 'type' column)
numerical_data = data.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
correlation_matrix = numerical_data.corr()

# Display the correlation matrix
print(correlation_matrix)

One-Hot Encoding

In [ ]:
# Perform one-hot encoding on the 'type' column
data_encoded = pd.get_dummies(data, columns=['type'], drop_first=True)

# Select all numerical columns (including the one-hot encoded 'type' columns)
numerical_data = data_encoded.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
correlation_matrix = numerical_data.corr()

# Display the correlation matrix
print(correlation_matrix)

Visualize the Correlation Matrix

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt

# Plot the correlation matrix as a heatmap
plt.figure(figsize=(8,8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

28. Plot the distribution of the amount column using a histogram.

In [ ]:
data['amount'].hist(bins=50)
plt.show()

29. Plot the distribution of the step column using a histogram.

In [ ]:
data['step'].hist(bins=50)
plt.show()

30. Plot the count of each transaction type using a bar plot.

In [ ]:
data['type'].value_counts().plot(kind='bar')
plt.show()

## 4) Feature Engineering

31. Create a new column isHighAmount where transactions above $1,000,000 are flagged as 1, else 0.

In [ ]:
data['isHighAmount'] = data['amount'].apply(lambda x: 1 if x > 1000000 else 0)
data['isHighAmount']

32. Create a new column isBalanceChangeOrigNegative where the originator's balance change is negative.

In [ ]:
data['isBalanceChangeDestNegative'] = data['balanceChangeDest'].apply(lambda x: 1 if x < 0 else 0)
data['isBalanceChangeDestNegative']

33. Create a new column isBalanceChangeDestNegative where the destination's balance change is negative.

In [ ]:
data['isBalanceChangeDestNegative'] = data['balanceChangeDest'].apply(lambda x: 1 if x < 0 else 0)
data['isBalanceChangeDestNegative']

34. Encode the type column using one-hot encoding.

In [ ]:
data = pd.get_dummies(data, columns=['type'], drop_first=True)

35. Normalize the amount column using Min-Max scaling.

In [ ]:
from sklearn.preprocessing import MinMaxScaler
scaler = MinMaxScaler()
data['amount_normalized'] = scaler.fit_transform(data[['amount']])
data['amount_normalized']

36. Count transactions between sender and receiver

In [ ]:
# Count transactions between sender and receiver
transaction_counts = data.groupby(['sender', 'receiver']).size().reset_index(name='transactionCount')
print(transaction_counts)

37. Create a new column transactionDuration representing the time difference between consecutive transactions.

In [ ]:
data['transactionDuration'] = data['step'].diff()
data['transactionDuration']

38. Create a new column isLargeBalanceChangeOrig where the originator's balance change is above $500,000.

In [ ]:
data['isLargeBalanceChangeOrig'] = data['balanceChangeOrig'].apply(lambda x: 1 if abs(x) > 500000 else 0)
data['isLargeBalanceChangeOrig']

39. Create a new column isLargeBalanceChangeDest where the destination's balance change is above $500,000.

In [ ]:
data['isLargeBalanceChangeDest'] = data['balanceChangeDest'].apply(lambda x: 1 if abs(x) > 500000 else 0)
data['isLargeBalanceChangeDest']

40. Create a new column isZeroBalanceOrig where the originator's new balance is zero.

In [ ]:
data['isZeroBalanceOrig'] = data['newbalanceOrig'].apply(lambda x: 1 if x == 0 else 0)
data['isZeroBalanceOrig']

## 5) Advanced Data Analysis

41. Find the top 10 senders (nameOrig) with the highest total transaction amounts.

In [ ]:
data.groupby('sender')['amount'].sum().nlargest(10)

42. Find the top 10 receivers (nameDest) with the highest total transaction amounts.

In [ ]:
data.groupby('receiver')['amount'].sum().nlargest(10)

43. Find the average transaction amount for each transaction type.

In [ ]:
# Reverse one-hot encoding to get the original 'type' column
data['type'] = data[['type_CASH_OUT', 'type_DEBIT', 'type_PAYMENT', 'type_TRANSFER']].idxmax(axis=1)

# Remove the prefix 'type_' from the values
data['type'] = data['type'].str.replace('type_', '')

# Now group by 'type' and calculate the mean amount
mean_amount_by_type = data.groupby('type')['amount'].mean()

# Display the results
print(mean_amount_by_type)

44. Find the total number of transactions per step (time unit).

In [ ]:
data['step'].value_counts().sort_index()

45. Find the total transaction amount per step (time unit).

In [ ]:
data.groupby('step')['amount'].sum()

46. Find the percentage of fraudulent transactions for each transaction type.

In [ ]:
data.groupby('type')['isFraud'].mean() * 100

47. Find the average originator balance (oldbalanceOrg) for fraudulent vs. non-fraudulent transactions.

In [ ]:
data.groupby('isFraud')['oldbalanceOrg'].mean()

48. Find the average destination balance (oldbalanceDest) for fraudulent vs. non-fraudulent transactions.

In [ ]:
data.groupby('isFraud')['oldbalanceDest'].mean()

49. Find the correlation between amount and isFraud.

In [ ]:
data['amount'].corr(data['isFraud'])

50. Find the correlation between oldbalanceOrg and newbalanceOrig.

In [ ]:
data['oldbalanceOrg'].corr(data['newbalanceOrig'])

## 6) Data Visualization

51. Plot the distribution of transaction amounts for fraudulent vs. non-fraudulent transactions.

In [ ]:
sns.boxplot(x='isFraud', y='oldbalanceOrg', data=data)
plt.show()

52. Plot the distribution of oldbalanceOrg for fraudulent vs. non-fraudulent transactions.

In [ ]:
sns.boxplot(x='isFraud', y='oldbalanceOrg', data=data)
plt.show()

53. Plot the distribution of oldbalanceDest for fraudulent vs. non-fraudulent transactions.

In [ ]:
sns.boxplot(x='isFraud', y='oldbalanceDest', data=data)
plt.show()

54. Plot the count of fraudulent transactions over time (step).

In [ ]:
data[data['isFraud'] == 1]['step'].value_counts().sort_index().plot()
plt.show()

55. Plot the total transaction amount over time (step).

In [ ]:
data.groupby('step')['amount'].sum().plot()
plt.show()

56. Plot a heatmap of the correlation matrix for numerical columns.

In [ ]:
# Select only numerical columns
numerical_data = data.select_dtypes(include=['int64', 'float64'])

# Compute correlation matrix
correlation_matrix = numerical_data.corr()

# Plot the correlation matrix as a heatmap
import seaborn as sns
import matplotlib.pyplot as plt

plt.figure(figsize=(10,8))
sns.heatmap(correlation_matrix, annot=True, cmap='coolwarm', fmt='.2f')
plt.title('Correlation Matrix')
plt.show()

57. Plot the distribution of transaction types for fraudulent transactions.

In [ ]:
data[data['isFraud'] == 1]['type'].value_counts().plot(kind='bar')
plt.show()

58. Plot the distribution of transaction types for non-fraudulent transactions.

In [ ]:
data[data['isFraud'] == 0]['type'].value_counts().plot(kind='bar')
plt.show()

59. Plot the distribution of balanceChangeOrig for fraudulent vs. non-fraudulent transactions.

In [ ]:
sns.boxplot(x='isFraud', y='balanceChangeOrig', data=data)
plt.show()

60. Plot the distribution of balanceChangeDest for fraudulent vs. non-fraudulent transactions.

In [ ]:
sns.boxplot(x='isFraud', y='balanceChangeDest', data=data)
plt.show()

## 7) Machine Learning Preparation

In [ ]:
data.head()

# Feature Selection

61. Encoding Categorical Features

In [ ]:
# identify categorical columns
categorical_cols = ['sender','receiver','type']  # Replace with actual categorical columns

# apply Label Encoding
label_encoders = {}
for col in categorical_cols:
    le = LabelEncoder()
    data[col] = le.fit_transform(data[col])
    label_encoders[col] = le

62. Split the dataset into features (X) and target (y).

In [ ]:
X = data.drop(columns=['isFraud'])

y = data['isFraud']

63. Training a Decision Tree for Feature Selection

In [ ]:
from sklearn.tree import DecisionTreeRegressor

model = DecisionTreeRegressor(random_state=42)
model.fit(X, y)

feature_importances = pd.DataFrame({
    'Feature': X.columns,
    'Importance': model.feature_importances_
})

# sort by importance
feature_importances = feature_importances.sort_values(by='Importance', ascending=False)

print(feature_importances)

64. Automate Feature Selection

In [ ]:
# define threshold
threshold = 0.01 # keep features with importance > 1%

# select important features
selected_features = feature_importances[feature_importances['Importance'] > threshold]['Feature'].tolist()

# filter dataset
X_selected = X[selected_features]

In [ ]:
print(X_selected.columns)

In [ ]:
X_selected


In [ ]:
y

## Feature Scaling:

In [ ]:
from sklearn.preprocessing import StandardScaler

# Define numerical columns to scale
numerical_columns = ['balanceChangeOrig', 'amount', 'amount_normalized', 'oldbalanceDest',
                     'newbalanceDest', 'type_PAYMENT', 'isZeroBalanceOrig',
                     'balanceChangeDest', 'sender', 'receiver', 'oldbalanceOrg',
                     'type_TRANSFER', 'type_encoded', 'step', 'isLargeBalanceChangeDest',
                     'type']

scaler = StandardScaler()

# Apply scaling
data[numerical_columns] = scaler.fit_transform(data[numerical_columns])


## Address Class Imbalance:

In [ ]:
from imblearn.over_sampling import SMOTE
from sklearn.utils import shuffle

# Resampling with SMOTE (Synthetic Minority Over-sampling Technique)
smote = SMOTE(random_state=42)
X_resampled, y_resampled = smote.fit_resample(X_selected, y)

# Shuffle the dataset after resampling
X_resampled, y_resampled = shuffle(X_resampled, y_resampled, random_state=42)

In [ ]:
X_resampled

In [ ]:
y_resampled

65. Split the dataset into training and testing sets (80% train, 20% test).

In [ ]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)

# MODEL SELECTION

1. Random forest classifier

In [ ]:
# Importing the model
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Initialize the model
model = RandomForestClassifier(random_state=42)

# Fit the model to the training data
model.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))
# Import necessary libraries
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get the model's predicted probabilities
y_prob = model.predict_proba(X_test)[:, 1]  # Probabilities for the positive class (fraud)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')  # Random classifier line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Get Precision and Recall values
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate the average precision score
average_precision = average_precision_score(y_test, y_prob)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'Precision-Recall curve (AP={average_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()

2. DecisionTreeClassifier

In [ ]:
# Initialize the model
model2 = DecisionTreeClassifier()

# Fit the model to the training data
model2.fit(X_train, y_train)

# Make predictions on the test set
y_pred = model.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred) * 100) 
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Import necessary libraries
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get the model's predicted probabilities
y_prob = model2.predict_proba(X_test)[:, 1]  # Probabilities for the positive class (fraud)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')  # Random classifier line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Get Precision and Recall values
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate the average precision score
average_precision = average_precision_score(y_test, y_prob)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'Precision-Recall curve (AP={average_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()


3. Logistic Regression (Classification)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Initialize the LogisticRegression model
model3 = LogisticRegression(random_state=42)

# Train the model
model3.fit(X_train, y_train)

# Make predictions
y_pred = model3.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Import necessary libraries
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get the model's predicted probabilities
y_prob = model3.predict_proba(X_test)[:, 1]  # Probabilities for the positive class (fraud)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')  # Random classifier line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Get Precision and Recall values
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate the average precision score
average_precision = average_precision_score(y_test, y_prob)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'Precision-Recall curve (AP={average_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()

4. K-Nearest Neighbors (KNN) for Classification

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import classification_report, confusion_matrix, accuracy_score

# Initialize the KNeighborsClassifier
model4 = KNeighborsClassifier()

# Train the model
model4.fit(X_train, y_train)

# Make predictions
y_pred = model4.predict(X_test)

# Evaluate the model
print("Accuracy:", accuracy_score(y_test, y_pred) * 100)
print("\nClassification Report:\n", classification_report(y_test, y_pred))
print("\nConfusion Matrix:\n", confusion_matrix(y_test, y_pred))

# Import necessary libraries
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc

# Get the model's predicted probabilities
y_prob = model4.predict_proba(X_test)[:, 1]  # Probabilities for the positive class (fraud)

# Compute ROC curve and AUC
fpr, tpr, thresholds = roc_curve(y_test, y_prob)
roc_auc = auc(fpr, tpr)

# Plot the ROC curve
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, color='blue', lw=2, label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', lw=2, linestyle='--')  # Random classifier line
plt.xlim([0.0, 1.0])
plt.ylim([0.0, 1.05])
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Get Precision and Recall values
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate the average precision score
average_precision = average_precision_score(y_test, y_prob)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'Precision-Recall curve (AP={average_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()

## Classification Model Comperison

In [ ]:
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# List of classification models to compare
models = {
    "Random Forest": RandomForestClassifier(random_state=42),
    "Logistic Regression": LogisticRegression(random_state=42),
    "K-Nearest Neighbors": KNeighborsClassifier(),
    "Decision Tree": DecisionTreeClassifier(random_state=42)
}

# Initialize a list to store accuracy scores
accuracy_scores = []

# Loop through each model, train, predict, and evaluate
for model_name, model in models.items():
    # Train the model
    model.fit(X_train, y_train)
    
    # Make predictions
    y_pred = model.predict(X_test)
    
    # Evaluate the model
    accuracy = accuracy_score(y_test, y_pred) * 100
    accuracy_scores.append(accuracy)

# Plotting the bar graph
plt.figure(figsize=(10, 6))
bars = plt.bar(models.keys(), accuracy_scores, color='skyblue')
plt.xlabel('Model')
plt.ylabel('Accuracy (%)')
plt.title('Comparison of Classification Models')
plt.xticks(rotation=45)

# Adding accuracy values on top of the bars
for bar, accuracy in zip(bars, accuracy_scores):
    yval = bar.get_height()
    plt.text(bar.get_x() + bar.get_width() / 2, yval + 1, f'{accuracy:.2f}%', ha='center', va='bottom')

plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Get Precision and Recall values
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate the average precision score
average_precision = average_precision_score(y_test, y_prob)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'Precision-Recall curve (AP={average_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()


1. Model Evaluation and Tuning:

In [ ]:
from sklearn.model_selection import cross_val_score, StratifiedKFold
from sklearn.metrics import confusion_matrix, classification_report, roc_auc_score
import matplotlib.pyplot as plt
import seaborn as sns

# Cross-validation
cv = StratifiedKFold(n_splits=5)
cv_scores = cross_val_score(model, X_resampled, y_resampled, cv=cv, scoring='roc_auc')  # Use ROC-AUC for fraud detection
print("Cross-Validation Scores:", cv_scores)
print("Average Cross-Validation Score:", cv_scores.mean())

# Evaluate performance on test set
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]  # Probabilities for ROC-AUC

# Confusion Matrix
cm = confusion_matrix(y_test, y_pred)
sns.heatmap(cm, annot=True, fmt="d", cmap='Blues', xticklabels=["Not Fraud", "Fraud"], yticklabels=["Not Fraud", "Fraud"])
plt.title("Confusion Matrix")
plt.show()

# Performance Metrics
print("Classification Report:\n", classification_report(y_test, y_pred))
print("ROC-AUC Score:", roc_auc_score(y_test, y_prob))

import pandas as pd

# Create a DataFrame to compare predictions with actual values
results_df = pd.DataFrame({
    'Actual': y_test,
    'Predicted': y_pred,
    'Probability': y_prob
})

# Function to classify prediction outcomes
def prediction_type(row):
    if row['Actual'] == 1 and row['Predicted'] == 1:
        return 'True Positive'
    elif row['Actual'] == 0 and row['Predicted'] == 1:
        return 'False Positive'
    elif row['Actual'] == 0 and row['Predicted'] == 0:
        return 'True Negative'
    elif row['Actual'] == 1 and row['Predicted'] == 0:
        return 'False Negative'

# Apply the classification
results_df['Prediction Type'] = results_df.apply(prediction_type, axis=1)

# Display counts of each prediction type
print(results_df['Prediction Type'].value_counts())

# Optionally display a sample of each type
for pred_type in ['True Positive', 'False Positive', 'True Negative', 'False Negative']:
    print(f"\nSample of {pred_type}s:")
    print(results_df[results_df['Prediction Type'] == pred_type].head())

#Visualization with Seaborn
sns.countplot(data=results_df, x='Prediction Type', palette='Set2')
plt.title("Count of Prediction Types")
plt.ylabel("Count")
plt.xlabel("Prediction Type")
plt.xticks(rotation=45)
plt.show()


Monitoring and Maintenance (Example of Model Performance Over Time):

In [ ]:
# Create a log file to store model performance over time
log_data = {'timestamp': [], 'roc_auc': [], 'accuracy': []}

# After each model evaluation, log the performance metrics
log_data['timestamp'].append(pd.Timestamp.now())
log_data['roc_auc'].append(roc_auc_score(y_test, y_prob))
log_data['accuracy'].append(model.score(X_test, y_test))

# Save to a CSV for historical tracking
log_df = pd.DataFrame(log_data)
log_df.to_csv('model_performance_log.csv', index=False)

# Monitor for drift: if performance drops significantly, trigger retraining
performance_threshold = 0.85  # Example threshold for ROC-AUC
if log_df['roc_auc'].iloc[-1] < performance_threshold:
    print("Model performance below threshold. Consider retraining.")

# Train  AutoGluon Model

 

Import Libraries 

In [ ]:
from autogluon.tabular import TabularDataset, TabularPredictor
from sklearn.model_selection import train_test_split
import pandas as pd

Split into train and test sets

In [ ]:
# Split into train and test sets (80% training, 20% testing)
X_train, X_test, y_train, y_test = train_test_split(X_resampled, y_resampled, test_size=0.3, random_state=42)
print(f"Training samples: {len(X_train)}")
print(f"Testing samples: {len(X_test)}")

Train the model using TabularPredictor

In [ ]:

# Combine X_train and y into a single DataFrame
X_train['isFraud'] =  y_resampled  # Add the target variable to the feature set

# Define the target column as a string
target = 'isFraud'

# Train the model using TabularPredictor
predictor1 = TabularPredictor(label=target).fit(
    train_data=X_train,
    time_limit=120,  # 2 minutes for quick results (increase for better accuracy)
    presets= 'best_quality'    #  Options: 'best_quality','medium_quality', 'high_quality' (faster vs. slower)
)


Evaluate the performance

In [ ]:
# Add the target column back to X_test
X_test['isFraud'] = y_test  # y_test should contain the true target labels

# Now evaluate the performance
performance = predictor1.evaluate(X_test)
print(f"Model Accuracy: {performance['accuracy']:.2f}")


In [ ]:
import matplotlib.pyplot as plt
from sklearn.metrics import roc_curve, auc
from autogluon.tabular import TabularPredictor

# Add the target column (y_test) back to X_test
X_test['isFraud'] = y_test  # y_test should contain the true target labels

# Now evaluate the performance
performance = predictor1.evaluate(X_test)
print(f"Model Accuracy: {performance['accuracy']:.2f}")

# Get the predicted probabilities from the model
y_prob = predictor1.predict_proba(X_test)

# Check the type of y_prob (it could be a DataFrame or ndarray)
print(f"y_prob type: {type(y_prob)}")
print(f"y_prob columns: {y_prob.columns if isinstance(y_prob, pd.DataFrame) else 'Not DataFrame'}")

# If y_prob is a DataFrame, access the second column (class 1 probabilities)
if isinstance(y_prob, pd.DataFrame):
    # Check if there are multiple columns (for binary classification)
    if y_prob.shape[1] > 1:
        y_prob = y_prob.iloc[:, 1].values  # Extract the second column (class 1 probabilities)
    else:
        y_prob = y_prob.iloc[:, 0].values  # For single-class (unlikely, but handle this case)

# If y_prob is a NumPy array, directly slice to get probabilities for class 1
elif isinstance(y_prob, np.ndarray):
    if y_prob.ndim > 1 and y_prob.shape[1] > 1:
        y_prob = y_prob[:, 1]  # Extract probabilities for class 1

# Compute ROC curve and AUC
fpr, tpr, _ = roc_curve(y_test, y_prob)  # y_test contains the true labels
roc_auc = auc(fpr, tpr)

# Plot ROC curve
plt.figure(figsize=(10, 6))
plt.plot(fpr, tpr, color='b', label=f'ROC curve (AUC = {roc_auc:.2f})')
plt.plot([0, 1], [0, 1], color='gray', linestyle='--', label='Random classifier')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('Receiver Operating Characteristic (ROC) Curve')
plt.legend(loc='lower right')
plt.show()

import matplotlib.pyplot as plt
from sklearn.metrics import precision_recall_curve
from sklearn.metrics import average_precision_score

# Get Precision and Recall values
precision, recall, thresholds = precision_recall_curve(y_test, y_prob)

# Calculate the average precision score
average_precision = average_precision_score(y_test, y_prob)

# Plot the Precision-Recall curve
plt.figure(figsize=(8, 6))
plt.plot(recall, precision, color='blue', label=f'Precision-Recall curve (AP={average_precision:.2f})')
plt.xlabel('Recall')
plt.ylabel('Precision')
plt.title('Precision-Recall Curve')
plt.legend(loc='best')
plt.grid(True)
plt.show()

leaderboard

In [ ]:
leaderboard = predictor1.leaderboard(X_test)
print(leaderboard)

In [ ]:
import matplotlib.pyplot as plt

# Assuming you already have the leaderboard dataframe
# Extract model names and corresponding test scores
models = leaderboard['model']
scores_test = leaderboard['score_test']

# Sort the leaderboard by score_test in descending order to have the best models on top
leaderboard_sorted = leaderboard.sort_values('score_test', ascending=False)

# Extract sorted model names and test scores
models_sorted = leaderboard_sorted['model']
scores_test_sorted = leaderboard_sorted['score_test']

# Create a bar graph
plt.figure(figsize=(10, 6))
bars = plt.barh(models_sorted, scores_test_sorted, color='skyblue')

# Add the accuracy text on each bar
for bar in bars:
    plt.text(bar.get_width(), bar.get_y() + bar.get_height() / 2,
             f'{bar.get_width():.4f}', va='center', ha='left', fontsize=10)

# Set labels and title
plt.xlabel('Test Score (Accuracy)')
plt.title('Model Comparison: Test Accuracy (Sorted)')
plt.gca().invert_yaxis()  # Best model at the top
plt.tight_layout()

# Display the plot
plt.show()


Generate predictions using the trained model

In [ ]:
import pandas as pd
from sklearn.metrics import accuracy_score, classification_report

# Generate predictions using the trained model (make sure X_test contains the features)
y_pred = predictor1.predict(X_test)

# Actual values (ground truth)
y_actual = X_test[target]  # Assuming `target` is the name of the target variable (e.g., 'isFraud')

# Compare predictions and actual values
comparison_df = pd.DataFrame({
    'Actual': y_actual,
    'Predicted': y_pred,
    'Correct': y_actual == y_pred  # A boolean column showing whether the prediction is correct
})

# Display comparison
print(comparison_df)  # Show the first few rows

# Calculate and print accuracy (just in case you want it separately)
accuracy = accuracy_score(y_actual, y_pred) * 100
print(f"Accuracy: {accuracy:.4f}")

# Detailed classification report (precision, recall, f1-score, etc.)
print("\nClassification Report:")
print(classification_report(y_actual, y_pred))

# Show the false predictions (where prediction is incorrect)
false_predictions_df = comparison_df[comparison_df['Correct'] == False]

# Display false predictions
print("\nFalse Predictions:")
print(false_predictions_df)
print(f"\nFalse Predictions: {len(false_predictions_df)}")
